In [ ]:
import os
# proxy = 'http://127.0.0.1:7890'
proxy = 'http://10.20.38.38:7890'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
os.environ["CUDA_VISIBLE_DEVICES"] = '6'
import open_clip

import torch
from torchvision.transforms import ToTensor
from PIL import Image
from tqdm import tqdm
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from extended_diffusers import ExtendedStableDiffusionXLPipeline
from pseudo_target_model import PseudoTargetModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"

# model_id = "stabilityai/stable-diffusion-xl-base-1.0"
# # model_id = "stabilityai/sdxl-turbo"
# unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
# unet.load_state_dict(load_file(hf_hub_download("ByteDance/SDXL-Lightning", "sdxl_lightning_8step_unet.safetensors")))
# vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
# scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler",timestep_spacing="trailing")
# pipe = ExtendedStableDiffusionXLPipeline.from_pretrained(model_id, unet=unet, vae=vae, scheduler=scheduler, torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
# pipe.vae.enable_slicing()
# pipe = pipe.to("cuda")

# Assuming you've already set up your pipeline as shown in your code
image_folder = '/mnt/dataset0/jiahua/closed-loop/imageset'

text_list = sorted(os.listdir(image_folder))

# List to store all latents
all_latents = []

# Define batch size outside the loop
batch_size = 4  # Adjust this value based on your GPU memory
target_size = (1024, 1024)  # Set according to your VAE model requirements


/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pyt

In [2]:
model_type = 'ViT-H-14'     
vlmodel, preprocess_train, feature_extractor = open_clip.create_model_and_transforms(
    model_type, pretrained='laion2b_s32b_b79k', precision='fp32', device=device)
    

In [3]:
def ImageEncoder(category_path, images, category):
    batch_size = 4   
    image_features_list = []
    
    # for i in range(0, len(images), batch_size):
    for i in tqdm(range(0, len(images), batch_size), desc=f"Processing {category}", leave=False):
        batch_images = images[i:i + batch_size]
        image_inputs = torch.stack([preprocess_train(Image.open(os.path.join(category_path, img)).convert("RGB")) for img in batch_images])

        with torch.no_grad():
            batch_image_features = vlmodel.encode_image(image_inputs.to(device))
        image_features_list.append(batch_image_features)
        del batch_images

    image_features = torch.cat(image_features_list, dim=0)                        
    return image_features

In [4]:
for category in tqdm(text_list, desc="Processing categories"):
    category_path = os.path.join(image_folder, category)
    
    if not os.path.isdir(category_path):
        continue
    
    image_files = sorted([f for f in os.listdir(category_path) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    batch_latents = ImageEncoder(category_path, image_files, category)
    # Append batch results to the overall list

    all_latents.append(batch_latents)        
       

Processing categories: 100%|██████████| 50/50 [01:06<00:00,  1.32s/it]


In [5]:
len(all_latents)

50

In [6]:
all_latents_tensor = torch.cat(all_latents, dim=0)

all_latents_tensor.shape

torch.Size([600, 1024])

In [7]:
output_dir = f"/home/ldy/Closed_loop_optimizing/data/clip_embed/open_clip/"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "600_image_embeds.pt")
torch.save(all_latents_tensor.cpu(), output_path)

In [13]:
# image_gt_folder = ['/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00014_bike/bike_14s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00181_television/television_14n.jpg'
#                         , '/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00177_t-shirt/t-shirt_13s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00135_pie/pie_15s.jpg'
#                         ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00131_pear/pear_13s.jpg']
image_gt_folder = ['/mnt/dataset1/ldy/datasets/datasets/THINGS_EEG/images_set/test_images/00056_crib/crib_14n.jpg',
                   '/mnt/dataset1/ldy/datasets/datasets/THINGS_EEG/images_set/test_images/00039_cheetah/cheetah_05s.jpg']

image_gt_path = image_gt_folder[1]
dir_name = os.path.basename(os.path.dirname(image_gt_path))  # '00014_bike'


In [14]:
batch_images = [image_gt_path]
image_inputs = torch.stack([preprocess_train(Image.open(os.path.join(category_path, img)).convert("RGB")) for img in batch_images])

print(f"image_inputs {image_inputs.shape}")
with torch.no_grad():
    image_feature = vlmodel.encode_image(image_inputs.to(device))


image_inputs torch.Size([1, 3, 224, 224])


In [15]:
output_dir = f"/home/ldy/Closed_loop_optimizing/data/clip_embed/open_clip/"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"{dir_name}_image_embeds.pt")
torch.save(image_feature.cpu(), output_path)